# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: d:\ADC 1\AI engineering\echochamber-project-team3
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_04"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().iloc[:15]

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [7]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [8]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [22]:
SYSTEM_PROMPT = """
Ești un expert în analiză politică și analiza discursului pe rețelele sociale din România.
Sarcina ta este să analizezi comentarii politice, extrăgând informații structurate.
Trebuie să returnezi strict un obiect JSON valid, fără introducere și fără text suplimentar.
"""

USER_PROMPT_TEMPLATE = """
Analizează următorul comentariu politic în limba română și identifică elementele solicitate:
1. target: entitatea politică vizată
2. stance: poziția autorului față de țintă ("PRO", "CONTRA", "NEUTRU")
3. sentiment: starea emoțională generală ("POZITIV", "NEGATIV", "NEUTRU")
4. tone: tonul utilizat (ex: sarcastic, revoltat, obiectiv)
5. topic: tema principală (ex: alegeri, corupție)
6. interpretation_problem: dacă e greu de interpretat textul ("da", "nu")
7. reason: scurtă justificare în română

IMPORTANT: Răspunsul tău trebuie să fie EXCLUSIV un dicționar JSON valid.
Returnează JSON valid cu exact aceste chei (observă acoladele duble obligatorii pentru codul Python):
{{
    "target": "...",
    "stance": "...",
    "sentiment": "...",
    "tone": "...",
    "topic": "...",
    "interpretation_problem": "...",
    "reason": "..."
}}

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [19]:
from openai import OpenAI


client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


if not model.startswith("models/"):
    model = f"models/{model}"

In [23]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [ ]:
n_comments = 10

sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,"```json\n{\n ""target"": ""Călin Georgescu"",\n..."
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,"```json\n{\n ""target"": ""Călin Georgescu"",\n..."
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,"```json\n{\n ""target"": ""Autoritățile"",\n ..."
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,"```json\n{\n ""target"": ""Persoanele care se ..."
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,"```json\n{\n ""target"": ""Sistemul de protecț..."
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,"```json\n{\n ""target"": ""Sistemul judiciar/m..."
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","```json\n{\n ""target"": ""Noua Ordine Mondial..."
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...","```json\n{\n ""target"": ""Un hot corupt"",\n ..."
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...","```json\n{\n ""target"": ""Crin Antonescu"",\n ..."
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,"```json\n{\n ""target"": ""Firme din Galați"",\..."


# 9. Verificam rezultatele

In [25]:
results_df.model_output[0]

'```json\n{\n    "target": "Călin Georgescu",\n    "stance": "CONTRA",\n    "sentiment": "NEGATIV",\n    "tone": "sarcastic",\n    "topic": "politică",\n    "interpretation_problem": "nu",\n    "reason": "Comentariul exprimă o ironie subtilă, sugerând că președintele nu este potrivit nici măcar pentru o funcție minoră (\'președinte de scară de bloc\') și îl sfătuiește să nu fugă în Dubai, implicând o lipsă de încredere sau o critică la adresa intențiilor sale."\n}\n```'

In [26]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [27]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,Călin Georgescu,CONTRA,NEGATIV,sarcastic,politică,nu,"Comentariul exprimă o ironie subtilă, sugerând..."
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,Călin Georgescu,PRO,POZITIV,apreciativ,discurs politic,nu,Autorul își exprimă aprecierea față de Călin G...
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,Autoritățile,CONTRA,NEGATIV,revoltat,responsabilitate publică,nu,Autorul critică autoritățile pentru lipsa de i...
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,Persoanele care se află sau trec prin zona res...,NEUTRU,NEGATIV,îngrijorat,Siguranța într-o anumită zonă,nu,Comentariul exprimă îngrijorare pentru siguran...
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,Sistemul de protecție a copilului,CONTRA,NEGATIV,"revoltat, empatic","Protecția copilului, traumele copiilor",nu,Autorul își exprimă durerea și revolta față de...
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,Sistemul judiciar/magistrații,CONTRA,NEGATIV,revoltat,justiție/corupție,nu,Comentariul critică direct activitatea sistemu...
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","Noua Ordine Mondială, Agenda 2030",CONTRA,NEGATIV,revoltat,conspirație mondială,nu,Autorul exprimă o poziție clară de opoziție fa...
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...",Un hot corupt,CONTRA,NEGATIV,revoltat,corupție,nu,Comentariul exprimă o puternică opoziție față ...
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...",Crin Antonescu,CONTRA,NEGATIV,sarcastic,alegeri,nu,Comentariul sugerează ironie legată de procent...
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,Firme din Galați,CONTRA,NEGATIV,acuzator,corupție,nu,Comentariul sugerează o legătură suspectă într...


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [28]:
# Salvăm rezultatele în folderul outputs sub formă de CSV
parsed_df.to_csv(ROOT / "outputs" / f"{student_id}_prompt_outputs.csv", index=False, encoding="utf-8")
print("Fișierul CSV a fost salvat cu succes!")

Fișierul CSV a fost salvat cu succes!


Da, promptul separă corect sentimentul de poziționare. Principala provocare rămâne în cazul comentariilor foarte lungi care au ținte multiple. În acele situații, modelul tinde să aleagă doar prima țintă sau să le amestece, fiind necesară o structură JSON mai complexă în versiunile viitoare pentru a lista separat fiecare țintă cu stance-ul ei.